In [1]:
!pip install -q librosa scipy matplotlib ipython torch torchaudio
import math
import glob
import os

import numpy as np
import torchaudio
import torch as th
import torch.nn as nn
import torch.nn.functional as tf
import librosa.filters as filters

import librosa
import scipy as sp
import matplotlib.pyplot as plt
import IPython.display as ipd

from typing import Optional, Tuple
from packaging.version import Version

EPSILON = float(np.finfo(np.float32).eps)
TORCH_VERSION = th.__version__

if TORCH_VERSION >= "1.7":
    from torch.fft import fft as fft_func
    print(TORCH_VERSION)
else:
    pass
root_path = "/kaggle/input/datasets/garvs777/libri3mix"
dataset_train360 = torchaudio.datasets.LibriMix(
    root=root_path,
    num_speakers=3,
    mode="min",
    sample_rate=8000,
    subset="train-360"
)
dataset_train100 = torchaudio.datasets.LibriMix(
    root=root_path,
    num_speakers=3,
    mode="min",
    sample_rate=8000,
    subset="train-100"
)
dataset_dev = torchaudio.datasets.LibriMix(
    root=root_path,
    num_speakers=3,
    mode="min",
    sample_rate=8000,
    subset="dev"
)
dataset_test = torchaudio.datasets.LibriMix(
    root=root_path,
    num_speakers=3,
    mode="min",
    sample_rate=8000,
    subset="test"
)
import os, json
import soundfile as sf

def build_json(wav_dir, out_path):
    entries = []
    for fname in sorted(os.listdir(wav_dir)):
        if not fname.endswith(".wav"):
            continue
        fpath = os.path.join(wav_dir, fname)
        info = sf.info(fpath)
        entries.append([fpath, info.frames])
    with open(out_path, "w") as f:
        json.dump(entries, f, indent=2)
    print(".json file formed.")

base = "/kaggle/input/datasets/garvs777/libri3mix/Libri3Mix/wav8k/min/dev"
out_path = "/kaggle/working/"

# comment out if building for the first time

# build_json(f"{base}/mix_clean", f"{out_path}/mix_clean_dev.json")
# build_json(f"{base}/s1", f"{out_path}/s1_dev.json")
# build_json(f"{base}/s2", f"{out_path}/s2_dev.json")
# build_json(f"{base}/s3", f"{out_path}/s3_dev.json")
import os
import json
from tkinter.tix import Tree
import numpy as np
from typing import Any, Tuple
import soundfile as sf
import torch
from pytorch_lightning import LightningDataModule
from pytorch_lightning.core.mixins import HyperparametersMixin
from torch.utils.data import ConcatDataset, DataLoader, Dataset
from typing import Dict, Iterable, List, Iterator
from rich import print
from pytorch_lightning.utilities import rank_zero_only
import pytorch_lightning as pl


import pytorch_lightning as pl



def normalize_tensor_wav(wav_tensor, eps=1e-8, std=None):
    mean = wav_tensor.mean(-1, keepdim=True)
    if std is None:
        std = wav_tensor.std(-1, keepdim=True)
    return (wav_tensor - mean) / (std + eps)



class Libri3MixDataset(Dataset):
    def __init__(
        self,
        json_dir: str = "/kaggle/working/",
        n_src: int = 3,
        sample_rate: int = 8000,
        fps: int = 25,
        segment: float = 4.0,
        normalize_audio: bool = False,
        audio_only: bool = True,
    ) -> None:
        super().__init__()
        self.EPS = 1e-8
        if json_dir is None:
            raise ValueError("JSON DIR is None!")
        if n_src < 1:
            raise ValueError("n_src must be >= 1, got {}".format(n_src))
        self.json_dir = json_dir
        self.sample_rate = sample_rate
        self.normalize_audio = normalize_audio
        self.audio_only = audio_only
        if segment is None:
            self.seg_len = None
            self.fps_len = None
        else:
            self.seg_len = int(segment * sample_rate)
            self.fps_len = int(segment * fps)
        self.n_src = n_src
        self.test = self.seg_len is None

        mix_json = os.path.join(json_dir, "mix_clean.json")
        # dynamically build s1.json ... sN.json instead of hardcoding names
        source_names = [f"s{i+1}" for i in range(n_src)]
        sources_json = [
            os.path.join(json_dir, name + ".json") for name in source_names
        ]

        with open(mix_json, "r") as f:
            mix_infos = json.load(f)
        sources_infos = []
        for src_json in sources_json:
            with open(src_json, "r") as f:
                sources_infos.append(json.load(f))

        self.mix = []
        self.sources = []

        if self.n_src == 1:
            # special case: each source becomes its own single-source example
            orig_len = len(mix_infos) * len(sources_infos)
            drop_utt, drop_len = 0, 0
            if not self.test:
                for i in range(len(mix_infos) - 1, -1, -1):
                    if mix_infos[i][1] < self.seg_len:
                        drop_utt += 1
                        drop_len += mix_infos[i][1]
                        del mix_infos[i]
                        for src_inf in sources_infos:
                            del src_inf[i]
                    else:
                        for src_inf in sources_infos:
                            self.mix.append(mix_infos[i])
                            self.sources.append(src_inf[i])
            else:
                for i in range(len(mix_infos)):
                    for src_inf in sources_infos:
                        self.mix.append(mix_infos[i])
                        self.sources.append(src_inf[i])

            print(
                "Drop {} utts({:.2f} h) from {} (shorter than {} samples)".format(
                    drop_utt, drop_len / sample_rate / 3600, orig_len, self.seg_len
                )
            )
            self.length = len(self.mix)

        else:
            # n_src >= 2 (2, 3, 4, ...): utterances stay index-aligned across all sources
            orig_len = len(mix_infos)
            drop_utt, drop_len = 0, 0
            if not self.test:
                for i in range(len(mix_infos) - 1, -1, -1):  # go backward
                    if mix_infos[i][1] < self.seg_len:
                        drop_utt += 1
                        drop_len += mix_infos[i][1]
                        del mix_infos[i]
                        for src_inf in sources_infos:
                            del src_inf[i]

            print(
                "Drop {} utts({:.2f} h) from {} (shorter than {} samples)".format(
                    drop_utt, drop_len / sample_rate / 3600, orig_len, self.seg_len
                )
            )
            self.mix = mix_infos
            self.sources = sources_infos
            self.length = len(self.mix)

    def __len__(self):
        return self.length

    def preprocess_audio_only(self, idx: int):
        if self.n_src == 1:
            if self.mix[idx][1] == self.seg_len or self.test:
                rand_start = 0
            else:
                rand_start = np.random.randint(0, self.mix[idx][1] - self.seg_len)
            stop = None if self.test else rand_start + self.seg_len

            x, _ = sf.read(
                self.mix[idx][0], start=rand_start, stop=stop, dtype="float32"
            )
            s, _ = sf.read(
                self.sources[idx][0], start=rand_start, stop=stop, dtype="float32"
            )

            target = torch.from_numpy(s)
            mixture = torch.from_numpy(x)
            if self.normalize_audio:
                m_std = mixture.std(-1, keepdim=True)
                mixture = normalize_tensor_wav(mixture, eps=self.EPS, std=m_std)
                target = normalize_tensor_wav(target, eps=self.EPS, std=m_std)
            return mixture, target.unsqueeze(0), self.mix[idx][0].split("/")[-1]

        # n_src >= 2 — generalized, works for any number of sources
        if self.mix[idx][1] == self.seg_len or self.test:
            rand_start = 0
        else:
            rand_start = np.random.randint(0, self.mix[idx][1] - self.seg_len)
        stop = None if self.test else rand_start + self.seg_len

        x, _ = sf.read(self.mix[idx][0], start=rand_start, stop=stop, dtype="float32")

        source_arrays = []
        for src in self.sources:
            s, _ = sf.read(src[idx][0], start=rand_start, stop=stop, dtype="float32")
            source_arrays.append(s)
        sources = torch.from_numpy(np.vstack(source_arrays))
        mixture = torch.from_numpy(x)

        if self.normalize_audio:
            m_std = mixture.std(-1, keepdim=True)
            mixture = normalize_tensor_wav(mixture, eps=self.EPS, std=m_std)
            sources = normalize_tensor_wav(sources, eps=self.EPS, std=m_std)

        return mixture, sources, self.mix[idx][0].split("/")[-1]

    def __getitem__(self, index: int):
        return self.preprocess_audio_only(index)

import pytorch_lightning as pl
class Libri3MixDataModule(pl.LightningDataModule):
    def __init__(
        self,
        train_dir: str,
        valid_dir: str,
        test_dir: str,
        n_src: int = 3,
        sample_rate: int = 8000,
        fps: int = 25,
        segment: float = 4.0,
        normalize_audio: bool = False,
        batch_size: int = 64,
        num_workers: int = 0,
        pin_memory: bool = False,
        persistent_workers: bool = False,
        audio_only: bool = True,
    ) -> None:
        super().__init__()
        if train_dir == None or valid_dir == None or test_dir == None:
            raise ValueError("JSON DIR is None!")
        if n_src < 1:
            raise ValueError("n_src must be >= 1, got {}".format(n_src))
        # this line allows to access init params with 'self.hparams' attribute
        self.train_dir = train_dir
        self.valid_dir = valid_dir
        self.test_dir = test_dir
        self.n_src = n_src
        self.sample_rate = sample_rate
        self.fps = fps
        self.segment = segment
        self.normalize_audio = normalize_audio
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.pin_memory = pin_memory
        self.persistent_workers = persistent_workers
        self.audio_only = audio_only
        self.data_train: Dataset = None
        self.data_val: Dataset = None
        self.data_test: Dataset = None

    def setup(self, stage=None):
        self.data_train = Libri3MixDataset(
            json_dir=self.train_dir,
            n_src=self.n_src,
            sample_rate=self.sample_rate,
            fps=self.fps,
            segment=self.segment,
            normalize_audio=self.normalize_audio,
            audio_only=self.audio_only,
        )
    
        self.data_val = Libri3MixDataset(
            json_dir=self.valid_dir,
            n_src=self.n_src,
            sample_rate=self.sample_rate,
            fps=self.fps,
            segment=self.segment,
            normalize_audio=self.normalize_audio,
            audio_only=self.audio_only,
        )
    
        self.data_test = Libri3MixDataset(
            json_dir=self.test_dir,
            n_src=self.n_src,
            sample_rate=self.sample_rate,
            fps=self.fps,
            segment=self.segment,
            normalize_audio=self.normalize_audio,
            audio_only=self.audio_only,
        )

    def train_dataloader(self) -> DataLoader:
        return DataLoader(
            dataset=self.data_train,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            persistent_workers=self.persistent_workers,
            pin_memory=self.pin_memory,
            drop_last=True,
        )

    def val_dataloader(self) -> DataLoader:
        return DataLoader(
            dataset=self.data_val,
            shuffle=False,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            persistent_workers=self.persistent_workers,
            pin_memory=self.pin_memory,
            drop_last=True,
        )

    def test_dataloader(self) -> DataLoader:
        return DataLoader(
            dataset=self.data_test,
            shuffle=False,
            batch_size=self.batch_size,
            num_workers=self.num_workers,
            persistent_workers=self.persistent_workers,
            pin_memory=self.pin_memory,
            drop_last=True,
        )

    @property
    def make_loader(self):
        return self.train_dataloader(), self.val_dataloader(), self.test_dataloader()

    @property
    def make_sets(self):
        return self.data_train, self.data_val, self.data_test
        

2.10.0+cu128


/tmp/ipykernel_163/1600789713.py:84: DeprecationWarning: The Tix Tk extension is unmaintained, and the tkinter.tix wrapper module is deprecated in favor of tkinter.ttk
  from tkinter.tix import Tree


In [2]:
import os, json
import soundfile as sf

DATASET_ROOT = "/kaggle/input/datasets/garvs777/libri3mix/Libri3Mix/wav8k/min"
WORK_ROOT = "/kaggle/working"
N_SRC = 3
SPLITS = {
    "train-100": "train-100",
    "dev": "dev",
    "test": "test",
}

def build_split_json(split_dir_name, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    src_folder = os.path.join(DATASET_ROOT, split_dir_name)

    def scan(folder_name):
        folder = os.path.join(src_folder, folder_name)
        entries = []
        for fname in sorted(os.listdir(folder)):
            if not fname.endswith(".wav"):
                continue
            fpath = os.path.join(folder, fname)
            info = sf.info(fpath)
            entries.append([fpath, info.frames])
        return entries

    mix_entries = scan("mix_clean")
    with open(os.path.join(out_dir, "mix_clean.json"), "w") as f:
        json.dump(mix_entries, f)

    for i in range(N_SRC):
        s_entries = scan(f"s{i+1}")
        with open(os.path.join(out_dir, f"s{i+1}.json"), "w") as f:
            json.dump(s_entries, f)

    print(f"[{split_dir_name}] mix={len(mix_entries)} utterances -> {out_dir}")

for split_dir_name in SPLITS.values():
    out_dir = os.path.join(WORK_ROOT, split_dir_name)
    build_split_json(split_dir_name, out_dir)

dm = Libri3MixDataModule(
    train_dir=os.path.join(WORK_ROOT, "train-100"),
    valid_dir=os.path.join(WORK_ROOT, "dev"),
    test_dir=os.path.join(WORK_ROOT, "test"),
    n_src=N_SRC,
    sample_rate=8000,
    segment=4.0,          # 4-second training chunks
    batch_size=2,         # keep small — bump up once you confirm memory headroom
    num_workers=2,
)
dm.setup()

mix=9300 utterances -> /kaggle/working/train-100

mix=3000 utterances -> /kaggle/working/dev

mix=3000 utterances -> /kaggle/working/test

Drop 669 utts(0.65 h) from 9300 (shorter than 32000 samples)

Drop 1265 utts(1.21 h) from 3000 (shorter than 32000 samples)

Drop 1515 utts(1.45 h) from 3000 (shorter than 32000 samples)

In [3]:
_train_loader = dm.train_dataloader()
_mixture, _sources, _names = next(iter(_train_loader))
print("mixture:", _mixture.shape)   # (B, T)
print("sources:", _sources.shape)   # (B, n_src, T)
print("example name:", _names[0])

import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

!pip install --no-index --find-links=/kaggle/input/datasets/garvs777/wheels/wheels \
    causal-conv1d einops 

mixture:
torch.Size([2, 32000])

sources:
torch.Size([2, 3, 32000])

example name: 6209-34599-0027_4830-25904-0008_3486-166424-0009.wav

2.10.0+cu128

12.8

True

Looking in links: /kaggle/input/datasets/garvs777/wheels/wheels
Processing /kaggle/input/datasets/garvs777/wheels/wheels/causal_conv1d-1.6.2.post1-cp312-cp312-linux_x86_64.whl


In [ ]:
class STFTEncoder(nn.Module):
    def __init__(self, n_fft=256, hop_length=64, win_length=256):
        super().__init__()
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.win_length = win_length
        self.register_buffer("window", th.hann_window(win_length))

    def forward(self, wav):
        # wav: (B, T)
        spec = th.stft(
            wav, n_fft=self.n_fft, hop_length=self.hop_length,
            win_length=self.win_length, window=self.window,
            return_complex=True, center=True,
        )  # (B, F, Tf) complex
        feat = th.stack([spec.real, spec.imag], dim=1)  # (B, 2, F, Tf)
        return feat, spec


class STFTDecoder(nn.Module):
    def __init__(self, n_fft=256, hop_length=64, win_length=256):
        super().__init__()
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.win_length = win_length
        self.register_buffer("window", th.hann_window(win_length))

    def forward(self, complex_spec, length=None):
        # complex_spec: (B, F, Tf) complex
        return th.istft(
            complex_spec, n_fft=self.n_fft, hop_length=self.hop_length,
            win_length=self.win_length, window=self.window,
            center=True, length=length,
        )

class BMamba(nn.Module):
    """Bidirectional block. Uses a BiGRU (no mamba_ssm dependency)."""
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.gru = nn.GRU(d_model, d_model // 2, batch_first=True, bidirectional=True)

    def forward(self, x):
        # x: (B, L, D)
        out, _ = self.gru(x)
        return self.norm(out + x)

class FDModule(nn.Module):
    """Frequency-axis BMamba, applied independently at every time frame."""
    def __init__(self, d_model):
        super().__init__()
        self.bmamba = BMamba(d_model)

    def forward(self, x):
        # x: (B, D, F, Tf)
        B, D, F, Tf = x.shape
        x = x.permute(0, 3, 2, 1).reshape(B * Tf, F, D)
        x = self.bmamba(x)
        x = x.reshape(B, Tf, F, D).permute(0, 3, 2, 1)
        return x


class TDModule(nn.Module):
    """Time-axis BMamba, applied independently at every frequency bin. Replaces BLSTM."""
    def __init__(self, d_model):
        super().__init__()
        self.bmamba = BMamba(d_model)

    def forward(self, x):
        # x: (B, D, F, Tf)
        B, D, F, Tf = x.shape
        x = x.permute(0, 2, 3, 1).reshape(B * F, Tf, D)
        x = self.bmamba(x)
        x = x.reshape(B, F, Tf, D).permute(0, 3, 1, 2)
        return x


class TFAModule(nn.Module):
    """Pools each frame across frequency into one embedding, runs full self-attention
    across frames for global context, then broadcasts the result back."""
    def __init__(self, d_model, n_heads=4):
        super().__init__()
        self.pool_proj = nn.Linear(d_model, d_model)
        self.mha = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.out_proj = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        # x: (B, D, F, Tf)
        B, D, F, Tf = x.shape
        frame_emb = x.mean(dim=2).permute(0, 2, 1)     # (B, Tf, D) avg-pool over freq
        frame_emb = self.pool_proj(frame_emb)
        attn_out, _ = self.mha(frame_emb, frame_emb, frame_emb)
        attn_out = self.out_proj(attn_out)              # (B, Tf, D)
        attn_out = attn_out.permute(0, 2, 1).unsqueeze(2)  # (B, D, 1, Tf)
        x = x + attn_out                                 # broadcast across frequency
        x = self.norm(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)
        return x


class SPMambaBlock(nn.Module):
    def __init__(self, d_model, n_heads=4):
        super().__init__()
        self.fd = FDModule(d_model)
        self.td = TDModule(d_model)
        self.tfa = TFAModule(d_model, n_heads)

    def forward(self, x):
        x = self.fd(x)
        x = self.td(x)
        x = self.tfa(x)
        return x

class SPMambaSeparator(nn.Module):
    def __init__(self, n_src=3, d_model=64, n_blocks=4, n_fft=256, hop_length=64,
                 win_length=256, n_heads=4):
        super().__init__()
        self.n_src = n_src
        self.encoder = STFTEncoder(n_fft, hop_length, win_length)
        self.decoder = STFTDecoder(n_fft, hop_length, win_length)
        self.input_proj = nn.Conv2d(2, d_model, kernel_size=3, padding=1)
        self.blocks = nn.ModuleList([SPMambaBlock(d_model, n_heads) for _ in range(n_blocks)])

        self.mask_head = nn.Conv2d(d_model, n_src * 2, kernel_size=3, padding=1)  # real+imag per slot
        self.presence_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, n_src),
        )

    def forward(self, wav):
        # wav: (B, T)
        feat, complex_spec = self.encoder(wav)     # feat (B,2,F,Tf), complex_spec (B,F,Tf)
        x = self.input_proj(feat)                  # (B, D, F, Tf)
        for block in self.blocks:
            x = block(x)

        B, D, F, Tf = x.shape
        masks = self.mask_head(x).view(B, self.n_src, 2, F, Tf)
        mask_complex = th.complex(masks[:, :, 0], masks[:, :, 1])   # (B, n_src, F, Tf)

        sep_specs = mask_complex * complex_spec.unsqueeze(1)        # (B, n_src, F, Tf)

        sep_waves = th.stack(
            [self.decoder(sep_specs[:, i], length=wav.shape[-1]) for i in range(self.n_src)],
            dim=1,
        )  # (B, n_src, T)

        presence_logits = self.presence_head(x)   # (B, n_src)
        return sep_waves, presence_logits

class ORPitExtractor(nn.Module):
    """Recursive one-and-rest extractor. One forward call splits the *current*
    mixture into (one source estimate, residual estimate). Run it n_src-1 times,
    feeding est_rest back in each time, to fully separate a fixed or unknown
    number of speakers."""
    def __init__(self, d_model=64, n_blocks=4, n_fft=256, hop_length=64,
                 win_length=256, n_heads=4):
        super().__init__()
        self.encoder = STFTEncoder(n_fft, hop_length, win_length)
        self.decoder = STFTDecoder(n_fft, hop_length, win_length)
        self.input_proj = nn.Conv2d(2, d_model, kernel_size=3, padding=1)
        self.blocks = nn.ModuleList([SPMambaBlock(d_model, n_heads) for _ in range(n_blocks)])
        # exactly 2 output slots: source, rest (real+imag each)
        self.mask_head = nn.Conv2d(d_model, 2 * 2, kernel_size=3, padding=1)

    def forward(self, wav):
        # wav: (B, T) -- the *current* mixture (original mix on step 0, residual after)
        feat, complex_spec = self.encoder(wav)     # feat (B,2,F,Tf), complex_spec (B,F,Tf)
        x = self.input_proj(feat)                  # (B, D, F, Tf)
        for block in self.blocks:
            x = block(x)

        B, D, F, Tf = x.shape
        masks = self.mask_head(x).view(B, 2, 2, F, Tf)
        mask_complex = th.complex(masks[:, :, 0], masks[:, :, 1])   # (B, 2, F, Tf)

        sep_specs = mask_complex * complex_spec.unsqueeze(1)        # (B, 2, F, Tf)
        sep_waves = th.stack(
            [self.decoder(sep_specs[:, i], length=wav.shape[-1]) for i in range(2)],
            dim=1,
        )  # (B, 2, T)

        est_source, est_rest = sep_waves[:, 0], sep_waves[:, 1]
        return est_source, est_rest


@th.no_grad()
def or_pit_infer(model, mixture, n_src):
    """Recursively extract n_src waveforms from a mixture at inference time.
    No ground truth available, so there's no PIT search here -- just peel off
    one source per step and keep going on the residual."""
    ests = []
    current_mix = mixture
    for _ in range(n_src - 1):
        est_src, est_rest = model(current_mix)
        ests.append(est_src)
        current_mix = est_rest
    ests.append(current_mix)  # whatever's left in the residual is the last speaker
    return th.stack(ests, dim=1)  # (B, n_src, T)

import itertools

def si_sdr(est, target, eps=1e-8):
    # est, target: (..., T)
    target_energy = th.sum(target ** 2, dim=-1, keepdim=True) + eps
    proj = th.sum(est * target, dim=-1, keepdim=True) * target / target_energy
    noise = est - proj
    ratio = th.sum(proj ** 2, dim=-1) / (th.sum(noise ** 2, dim=-1) + eps)
    return 10 * th.log10(ratio + eps)


def deterministic_energy_order(sources):
    # sources: (B, n_src, T) ground truth -> sorted loudest-first
    energy = th.sum(sources ** 2, dim=-1)
    order = th.argsort(energy, dim=-1, descending=True)
    return th.gather(sources, 1, order.unsqueeze(-1).expand(-1, -1, sources.shape[-1]))


def deterministic_sisdr_loss(est_waves, sources):
    target_sorted = deterministic_energy_order(sources)
    return -si_sdr(est_waves, target_sorted).mean()


def pit_sisdr_loss(est_waves, sources):
    # est_waves, sources: (B, n_src, T)
    n_src = est_waves.shape[1]
    best = None
    for perm in itertools.permutations(range(n_src)):
        score = si_sdr(est_waves, sources[:, perm, :]).mean(dim=-1)  # (B,)
        best = score if best is None else th.maximum(best, score)
    return -best.mean()


def presence_bce_loss(presence_logits, n_active):
    # presence_logits: (B, n_src); n_active: (B,) long tensor
    B, n_src = presence_logits.shape
    idx = th.arange(n_src, device=presence_logits.device).unsqueeze(0).expand(B, -1)
    target = (idx < n_active.unsqueeze(1)).float()
    return tf.binary_cross_entropy_with_logits(presence_logits, target)

def or_pit_loss(model, mixture, sources, use_estimated_residual=True):
    """Recursive one-and-rest PIT loss (Takahashi et al., 2019 style).
    mixture: (B, T) original mix. sources: (B, n_src, T) ground truth.
    Runs `model` n_src-1 times, each time picking whichever *remaining*
    ground-truth speaker best explains the current (source, rest) output pair,
    then recurses on the estimated residual. Returns a scalar loss (negative
    mean SI-SDR across all n_src assignments, per sample).
    """
    B, n_src, T = sources.shape
    active = th.ones(B, n_src, dtype=th.bool, device=sources.device)
    live_sources = sources.clone()  # zeroed out per-sample as speakers get assigned
    current_mix = mixture
    total_score = th.zeros(B, device=sources.device)
    n_steps = n_src - 1

    for _ in range(n_steps):
        est_src, est_rest = model(current_mix)  # (B, T) each

        total_sum = live_sources.sum(dim=1)                       # (B, T)
        src_score = si_sdr(est_src.unsqueeze(1), live_sources)     # (B, n_src)
        rest_targets = total_sum.unsqueeze(1) - live_sources       # (B, n_src, T)
        rest_score = si_sdr(est_rest.unsqueeze(1), rest_targets)   # (B, n_src)
        scores = (src_score + rest_score).masked_fill(~active, float("-inf"))

        best_score, best_idx = scores.max(dim=1)  # (B,), (B,)
        total_score = total_score + best_score

        # remove the chosen speaker (per-sample index) from the pool
        active.scatter_(1, best_idx.unsqueeze(1), False)
        live_sources = live_sources * active.unsqueeze(-1).float()

        chosen_rest_gt = th.gather(
            rest_targets, 1, best_idx.view(B, 1, 1).expand(-1, 1, T)
        ).squeeze(1)  # (B, T)
        # feed the model's own residual back in (matches inference); set
        # use_estimated_residual=False to teacher-force on the GT residual instead
        current_mix = est_rest if use_estimated_residual else chosen_rest_gt.detach()

    # exactly one speaker left per sample -> direct SI-SDR, no PIT needed
    last_idx = active.float().argmax(dim=1)
    final_target = th.gather(
        sources, 1, last_idx.view(B, 1, 1).expand(-1, 1, T)
    ).squeeze(1)
    total_score = total_score + si_sdr(current_mix, final_target)

    return -(total_score / n_src).mean()

import pytorch_lightning as pl

class SPMambaLitModule(pl.LightningModule):
    def __init__(self, n_src=3, d_model=64, n_blocks=4, lr=1e-3,
                 assignment_mode="or_pit", presence_weight=1.0):
        super().__init__()
        self.save_hyperparameters()
        if assignment_mode == "or_pit":
            self.model = ORPitExtractor(d_model=d_model, n_blocks=n_blocks)
        else:
            self.model = SPMambaSeparator(n_src=n_src, d_model=d_model, n_blocks=n_blocks)

    def forward(self, wav):
        if self.hparams.assignment_mode == "or_pit":
            # recursive one-and-rest inference -> (B, n_src, T), no presence logits
            return or_pit_infer(self.model, wav, self.hparams.n_src)
        return self.model(wav)

    def _step(self, batch, stage):
        mixture, sources, _ = batch  # mixture (B,T), sources (B,n_src,T)

        if self.hparams.assignment_mode == "or_pit":
            sep_loss = or_pit_loss(self.model, mixture, sources)
            loss = sep_loss
            self.log(f"{stage}_sep_loss", sep_loss, prog_bar=True, batch_size=mixture.shape[0])
            self.log(f"{stage}_loss", loss, prog_bar=True, batch_size=mixture.shape[0])
            return loss

        est_waves, presence_logits = self.model(mixture)

        if self.hparams.assignment_mode == "pit":
            sep_loss = pit_sisdr_loss(est_waves, sources)
        else:
            sep_loss = deterministic_sisdr_loss(est_waves, sources)

        n_active = th.full((mixture.shape[0],), self.hparams.n_src,
                            dtype=th.long, device=mixture.device)
        pres_loss = presence_bce_loss(presence_logits, n_active)

        loss = sep_loss + self.hparams.presence_weight * pres_loss
        self.log(f"{stage}_sep_loss", sep_loss, prog_bar=True, batch_size=mixture.shape[0])
        self.log(f"{stage}_presence_loss", pres_loss, prog_bar=True, batch_size=mixture.shape[0])
        self.log(f"{stage}_loss", loss, prog_bar=True, batch_size=mixture.shape[0])
        return loss

    def training_step(self, batch, batch_idx):
        return self._step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self._step(batch, "val")

    def configure_optimizers(self):
        return th.optim.Adam(self.parameters(), lr=self.hparams.lr)

import torch

print(torch.cuda.is_available())
print(torch.cuda.current_device() if torch.cuda.is_available() else "No GPU")
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
batch = next(iter(dm.train_dataloader()))

print(type(batch))
print(len(batch))

for i, item in enumerate(batch):
    print(i, type(item))
print(batch[2])
device = torch.device("cuda")

_sanity_model = SPMambaLitModule(
    n_src=N_SRC,
    d_model=64,
    n_blocks=2,
    assignment_mode="deterministic",
).to(device)

mixture, sources, filenames = next(iter(dm.train_dataloader()))

mixture = mixture.to(device)
sources = sources.to(device)

_loss = _sanity_model._step(
    (mixture, sources, filenames),
    "sanity"
)

print("Sanity loss:", _loss.item())
lit_model = SPMambaLitModule(
    n_src=N_SRC,
    d_model=64,
    n_blocks=4,
    lr=1e-3,
    assignment_mode="or_pit",   # "pit" still available as the comparison baseline
    presence_weight=1.0,
)

trainer = pl.Trainer(
    max_epochs=15,
    accelerator="auto",
    devices="auto",
    precision="16-mixed" if th.cuda.is_available() else 32,
    log_every_n_steps=10,
    gradient_clip_val=5.0,
)

trainer.fit(
    lit_model,
    train_dataloaders=dm.train_dataloader(),
    val_dataloaders=dm.val_dataloader(),
)

lit_model.eval()
_val_batch = next(iter(dm.val_dataloader()))
_mix, _src, _names = _val_batch

with th.no_grad():
    if lit_model.hparams.assignment_mode == "or_pit":
        _est_waves = lit_model(_mix.to(lit_model.device))   # (B, n_src, T), recursive extraction
        _presence_probs = None
    else:
        _est_waves, _presence_logits = lit_model.model(_mix.to(lit_model.device))
        _presence_probs = th.sigmoid(_presence_logits)

print("Example:", _names[0])
if _presence_probs is not None:
    print("Presence probabilities (slot -> prob active):", _presence_probs[0].cpu().numpy())



True

0

NVIDIA RTX PRO 6000 Blackwell Server Edition

<class 'list'>

3

...

┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type           ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ ORPitExtractor │  255 K │ train │     0 │
└───┴───────┴────────────────┴────────┴───────┴───────┘

Trainable params: 255 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 255 K                                                                                                
Total estimated model params size (MB): 1.022                                                                      
Modules in train mode: 66                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/tmp/ipykernel_163/1853715760.py:178: UserWarning: ComplexHalf support is experimental and many operators don't 
support it yet. (Triggered internally at /pytorch/aten/src/ATen/EmptyTensor.cpp:54.)
  mask_complex = th.complex(masks[:, :, 0], masks[:, :, 1])   # (B, 2, F, Tf)

In [8]:
print("Mixture:")
ipd.display(ipd.Audio(_mix[0].cpu().numpy(), rate=8000))

for i in range(N_SRC):
    print(f"Ground-truth speaker {i+1}:")
    ipd.display(ipd.Audio(_src[0, i].cpu().numpy(), rate=8000))
    label = f"Predicted slot {i+1}"
    if _presence_probs is not None:
        label += f" (presence={_presence_probs[0, i].item():.2f})"
    print(label + ":")
    ipd.display(ipd.Audio(_est_waves[0, i].cpu().numpy(), rate=8000))

Mixture:

Ground-truth speaker 1:

Predicted slot 1:

Ground-truth speaker 2:

Predicted slot 2:

Ground-truth speaker 3:

Predicted slot 3:

In [6]:
th.save(lit_model.model.state_dict(), "/kaggle/working/orpit_extractor_weights.pt")
trainer.save_checkpoint("/kaggle/working/orpit_full_checkpoint.ckpt")

from IPython.display import FileLink
FileLink("/kaggle/working/orpit_extractor_weights.pt")

`weights_only` was not set, defaulting to `False`.


/kaggle/working/orpit_extractor_weights.pt